# Nss Midcourse project: Creating pathfinder 2e monster encounter
### combine all input datasheets to create each character build

In [1]:
#Import libraries 
import pandas as pd
import numpy as np
import requests
import matplotlib as plt


import requests
from bs4 import BeautifulSoup
import pandas as pd
from io import StringIO
import re
import random

import warnings
from pandas.errors import SettingWithCopyWarning
warnings.simplefilter(action='ignore', category=FutureWarning)
warnings.simplefilter(action='ignore', category=SettingWithCopyWarning)

### The structure of a character sheet:
* ability scores: strength, dexterity, constitution, intelligence, wisdonm, charisma
    * ability modifiers= 8-9 = -1, 10-11 = 0, 12-13= +1, 14=15= +2, 16-17 = +3, 18-19= +4
* proficency:
  * untrained= +0
  * trained= 2+ level
  * expert= 4+ level
  * master = 6+ level
  * legendary = 8+ level
* Defenses: armor class, fortitude, reflex, will, hit points (HP + con)
* Equipment proficencies: armor proficencies, weapon proficencies (unarmored, light, medium, heavy)
* perception: wisdom + proficency
* Skills: Acrobatics, Arcana, Athletics, Crafting, Deception, Diplomacy, Intimidation, Medicine, Nature, Religion, Occultism, Performance, Society, Stealth, Survival, Thievery
      * simplify by removing skills
* feats
* spellcasting: magical tradiiton, spell attack, spell dc
* Character creation rules: up to 4 free boosts, ability cannot exceed 18 at level 1

### At character creation (base): 
* all ability scores are 10
* Everything is untrained

### Create a dataframe with all of these features as a row, and input the values depending on the build combination

In [2]:
character_sheet_rows= ['hit points', 'speed', 'perception', 'fortitude',  'reflex',  'will', 
                       'armor class', 'unarmored', 'light', 'medium', 'heavy', 
                       'unarmed',  'simple',  'martial',  'advanced',  'other', 
                       'magical tradition', 'spell attack',  'spell dc', ]

character_sheet= pd.DataFrame(character_sheet_rows, columns=['build metrics'])
character_sheet= character_sheet.set_index('build metrics', drop=True)
character_sheet

#add a column for each level
base_formula= ['class + ancestry + level + con', 'ancestry', 'wis+prof', 'con+prof',  'dex+prof', 'wis+prof', 
              '10+dex+prof',  'prof',  'prof', 'prof',  'prof',  'prof',  'prof',  'prof',  'prof', 'prof', 
               'class', 'mod+prof',  '10+mod+prof', ]

character_sheet['formula']= base_formula
character_sheet['ability_modifier']= ['con', 'none', 'wis',  'con',  'dex',  'wis', 
                                      'dex',  'prof',  'prof', 'prof', 'prof',  'prof',  'prof', 'prof',  'prof',  'prof',
                                      'class', 'spell ability', 'spell ability', ]

character_sheet['base_value']= [0, 0, 0, 0,  0, 0, 
                                      10,  0,  0, 0, 0,0, 0, 0, 0, 0,
                                      0, 0, 10 ]

#add in filler rows for the information from class and ancestry choices
character_sheet['ancestry_boost']= character_sheet.index
character_sheet['class_boost']= character_sheet.index
character_sheet['equipment_boost']= character_sheet.index

#save character sheet in base templates 
character_sheet.to_csv('../templates/character_sheet_template.csv')

#view dataframe
character_sheet

,formula,ability_modifier,base_value,ancestry_boost,class_boost,equipment_boost
build metrics,,,,,,
hit points,class + ancestry + level + con,con,0,hit points,hit points,hit points
speed,ancestry,none,0,speed,speed,speed
perception,wis+prof,wis,0,perception,perception,perception
fortitude,con+prof,con,0,fortitude,fortitude,fortitude
reflex,dex+prof,dex,0,reflex,reflex,reflex
will,wis+prof,wis,0,will,will,will
armor class,10+dex+prof,dex,10,armor class,armor class,armor class
unarmored,prof,prof,0,unarmored,unarmored,unarmored
light,prof,prof,0,light,light,light


## Simplifying character building:
* Base stats are ['strength','dexterity', 'constitution', 'intelligence', 'wisdom', 'charisma']
* At level one (character creation) you get ability boosts (+2) or flaws (-2) based on ancestry, class, background, and 4 free boosts to any of those stats
* To simplify having to choose the background and 4 free boosts -> just add +2 to every base stat to have the base start at 12
* For the code-> pull the boosts from the ancestry and class, then have the 6 free boosts based on calculating which stats need the biggests boosts to reach the max of 18
* Then teh ability modifier depends on the number range (stat-10 / 2)

### THEN apply the same calculations to the remaining base features 


In [3]:
##create the base dataframe for just the stats
stat_rows= ['strength', 'dexterity', 'constitution', 'intelligence', 'wisdom', 'charisma']

#make new dataframe 
base_abilities = pd.DataFrame(stat_rows, columns=['attribute'])
base_abilities

#create a column of the remaining possible sources of boosts 
base_abilities['base_value']= ['12','12', '12', '12', '12', '12']
base_abilities['ancestry_boost']= base_abilities['attribute']
base_abilities['class_boost']= base_abilities['attribute']

base_abilities

,attribute,base_value,ancestry_boost,class_boost
0,strength,12,strength,strength
1,dexterity,12,dexterity,dexterity
2,constitution,12,constitution,constitution
3,intelligence,12,intelligence,intelligence
4,wisdom,12,wisdom,wisdom
5,charisma,12,charisma,charisma


### Open ancestry attribute stats

In [4]:
ancestry_stat= pd.read_csv('../output_data/ancestry_attribute.csv')
ancestry_stat= ancestry_stat.drop('Unnamed: 0', axis=1)
ancestry_stat= ancestry_stat.set_index('Attribute', drop=True)
ancestry_stat.loc['Speed']= [x.split(' ')[0] for x in ancestry_stat.loc['Speed']]
ancestry_stat.loc['Speed'] = ancestry_stat.loc['Speed'].astype(int) #25- ancestry_stat.loc['Speed'].astype(int)

### add two different human and orc builds by chosing the base ability boosts to streamline 

ancestry_stat['human-str'] = ancestry_stat['human']

ancestry_stat['human-dex'] = ancestry_stat['human']
ancestry_stat['orc-str'] = ancestry_stat['orc']
ancestry_stat['orc-dex'] = ancestry_stat['orc']

#loop between the new columns to add in the ability boosts to the main stat AND constitution 
free_ability=['Strength', 'Strength', 'Constitution', 'Constitution','Dexterity', 'Dexterity', 'Constitution', 'Constitution']
free_ability_class=['human-str', 'orc-str','human-str', 'orc-str', 'human-dex', 'orc-dex', 'human-dex', 'orc-dex']

for row in range (len(free_ability_class)):
    ancestry_stat.at[free_ability[row], free_ability_class[row]] = '+2'

#drop the free boost column and the normal human and orc columns
ancestry_stat= ancestry_stat.drop(['Two free ability boosts'], axis=0)	
ancestry_stat= ancestry_stat.drop(['human', 'orc'], axis=1)	

#save character sheet in base templates 
ancestry_stat.to_csv('../templates/ancestry_stat_boost_template.csv')

#view dataframe 
ancestry_stat


,dwarf,elf,halfling,human-str,human-dex,orc-str,orc-dex
Attribute,,,,,,,
Hit Points,10,6,6,8,8,10,10
Speed,20,30,25,25,25,25,25
Strength,0,0,-2,+2,0,+2,0
Dexterity,0,+2,+2,0,+2,0,+2
Constitution,+2,-2,0,+2,+2,+2,+2
Intelligence,0,+2,0,0,0,0,0
Wisdom,+2,0,+2,0,0,0,0
Charisma,-2,0,0,0,0,0,0


### Open class stats

In [5]:
class_base_stat= pd.read_csv('../output_data/all_class_base_stats.csv')
class_base_stat= class_base_stat.set_index('Class', drop=True).drop(['Unnamed: 0'], axis=1)	
class_base_stat

#clean up primary ability column for easier future filtering
class_base_stat['Primary Ability']= [x.split('[')[-1] for x in class_base_stat['Primary Ability']] 
class_base_stat['Primary Ability']= [x.split(']')[0] for x in class_base_stat['Primary Ability']] 
class_base_stat['Primary Ability']
#Use explode to create new rows based on primary ability scores
class_base_stat['Primary Ability']= [re.sub(r'\[.*?\]', '', x) for x in class_base_stat['Primary Ability']] 
class_base_stat['Primary Ability']= [x.split(',') for x in class_base_stat['Primary Ability']]
class_base_stat= class_base_stat.explode('Primary Ability')

##add extra columns for filtering later
class_base_stat['ability boost'] = '+2'
class_base_stat['class_build']= class_base_stat.index + ' '+class_base_stat['Primary Ability'].str.replace("'", '')

#substitute in the 'All' metric in the defenses column 
all_armor= ['unarmored', 'light', 'medium', 'heavy']
defense_list= class_base_stat['Defenses'].str.split(",")

clean_defense_list= []
defense_list_expert= []

for item in defense_list:
    if 'All' in item:
        item= item + all_armor
    elif 'EXPERT' in item:
        print(item)
    clean_defense_list.append(item)
clean_defense_list

class_base_stat['Defenses']= clean_defense_list

#add a column for the EXPERT option in defenses and attacks 
class_base_stat['Defenses_expert'] = class_base_stat['Defenses']
class_base_stat['Attacks_expert']= class_base_stat['Attacks']

#save spreadsheet
class_base_stat

,Primary Ability,Hit Points per Level,Perception,Fortitude,Reflex,Will,Skills,Defenses,Attacks,Spells,Class/Spell DC,ability boost,class_build,Defenses_expert,Attacks_expert
Class,,,,,,,,,,,,,,,
Alchemist,'Intelligence',8,Trained,Expert,Expert,Trained,"['Crafting', '+3 of Choice']","[Light, Unarmored]","Simple, Unarmed, Alchemical Bombs",--,Trained,+2,Alchemist Intelligence,"[Light, Unarmored]","Simple, Unarmed, Alchemical Bombs"
Barbarian,'Strength',12,Expert,Expert,Trained,Expert,"['Athletics', '+3 of Choice']","[Light, Medium, Unarmored]","Simple, Martial, Unarmed",--,Trained,+2,Barbarian Strength,"[Light, Medium, Unarmored]","Simple, Martial, Unarmed"
Bard,'Charisma',8,Expert,Trained,Trained,Expert,"['Occultism and Performance', '+ 4 of Choice']","[Light, Unarmored]","Simple, Unarmed, Longsword, Rapier, Sap, Short...",Occult,Trained,+2,Bard Charisma,"[Light, Unarmored]","Simple, Unarmed, Longsword, Rapier, Sap, Short..."
Champion,'Strength ',10,Trained,Expert,Trained,Expert,"[""Religion and 1 of your Deity's"", '+2 of Choi...","[All, Unarmored, unarmored, light, medium, he...","Simple, Martial, Unarmed",Divine,Trained,+2,Champion Strength,"[All, Unarmored, unarmored, light, medium, he...","Simple, Martial, Unarmed"
Champion,' Dexterity',10,Trained,Expert,Trained,Expert,"[""Religion and 1 of your Deity's"", '+2 of Choi...","[All, Unarmored, unarmored, light, medium, he...","Simple, Martial, Unarmed",Divine,Trained,+2,Champion Dexterity,"[All, Unarmored, unarmored, light, medium, he...","Simple, Martial, Unarmed"
Cleric,'Wisdom',8,Trained,Trained,Trained,Expert,"[""Religion and 1 of your Deity's"", '+2 of Choi...","[Unarmored, Armor noted by Doctrine]","Simple, Deity's Favored Weapon, Unarmed",Divine,Trained,+2,Cleric Wisdom,"[Unarmored, Armor noted by Doctrine]","Simple, Deity's Favored Weapon, Unarmed"
Druid,'Wisdom',8,Trained,Trained,Trained,Expert,"['Nature and 1 from your Order', '+2 of Choice']","[Light, Medium, Unarmored]","Simple, Unarmed",Primal,Trained,+2,Druid Wisdom,"[Light, Medium, Unarmored]","Simple, Unarmed"
Fighter,'Strength ',10,Expert,Expert,Expert,Trained,"['Acrobatics OR Athletics', '+3 of Choice']","[All, Unarmored, unarmored, light, medium, he...","Advanced; EXPERT: Simple, Martial, Unarmed",--,Trained,+2,Fighter Strength,"[All, Unarmored, unarmored, light, medium, he...","Advanced; EXPERT: Simple, Martial, Unarmed"
Fighter,' Dexterity',10,Expert,Expert,Expert,Trained,"['Acrobatics OR Athletics', '+3 of Choice']","[All, Unarmored, unarmored, light, medium, he...","Advanced; EXPERT: Simple, Martial, Unarmed",--,Trained,+2,Fighter Dexterity,"[All, Unarmored, unarmored, light, medium, he...","Advanced; EXPERT: Simple, Martial, Unarmed"


### Iterate between all ancestry-class combinations and calculate the base attibute score

In [6]:
### Create a dataframe to store all output calculations for the ancestry-class combination
all_build_stats=pd.DataFrame(columns=['ancestry', 'class', 'level', 'strength','dexterity', 'constitution', 'intelligence', 'wisdom', 'charisma'])

#make a list of ancestry-class combinations?
all_ancestry= ancestry_stat.columns.to_list()
all_class_list= class_base_stat['class_build'].to_list()

chose_class_list=['Barbarian Strength', 'Fighter Strength ', 'Fighter   Dexterity', 'Rogue Dexterity ', 'Sorcerer Charisma']

#loop between each character combination to make a unique character sheet, and store it as a new data frame 
ancestry_stat['attribute_merge']= ancestry_stat.index.str.lower()
ancestry_stat= ancestry_stat.set_index('attribute_merge')

#make a loop

#first pull out the ancestry value 
for ancestry in all_ancestry:
    ancestry_choice= ancestry.lower().strip()
    
    #Then cycle through each class option for that ancestry 
    for build in range(len(chose_class_list)):
        class_choice= chose_class_list[build].lower().strip()
        build_name= (ancestry_choice +' ' + class_choice)

        #copy base ability dataframe 
        base_for_build= pd.DataFrame()

        #For the speciifc build combination, pull out all the information about ability calculations and save in a final dataframe 
        #Create the first variables for the final dataframe 
        build_list= [ancestry_choice, class_choice, 1]

        ####
        # Ancestry stat boosts
        ####
        #convert chosen ANCESTRY stats to a dictionary 
        ancestry_value= ancestry_stat[f'{ancestry_choice}'].to_dict()
        
        #replace the ancestry_boost values with the dictionary 
        base_for_build['ancestry_boost'] = base_abilities['ancestry_boost'].replace(ancestry_value)
        base_for_build['attribute']= base_abilities.index
        
        # ####
        # #CLass stat boosts
        # ####
        
        #convert chosen class stats to a dictionary
        class_merge= class_base_stat[class_base_stat['class_build'] == chose_class_list[build]].T
        class_merge['metrics']= class_merge.index.str.lower()
        class_merge= class_merge.set_index('metrics', drop=True)
        class_merge= class_merge.rename(index={'hit points per level': 'hit points'})
        class_value= class_merge.to_dict()
        
        #add the +2 for the class ability boost
        class_ability_boost= class_value[class_merge.columns[0]]['primary ability'].lower().replace("'", "").strip()
        #create a new dataframe with the updated attributes for the specific informaiton for the build
        base_for_build['class_boost']=base_abilities['class_boost'].replace(class_ability_boost, '+2')
        
        #####
        # Calculate final ability modifiers for the build
        #####
        base_for_build['ancestry_boost']= pd.to_numeric(base_for_build['ancestry_boost'], errors='coerce').fillna(0)
        base_for_build['class_boost']= pd.to_numeric(base_for_build['class_boost'], errors='coerce').fillna(0)
        base_for_build['base_value']= pd.to_numeric(base_abilities['base_value'], errors='coerce').fillna(0)
        base_for_build['total_boost']= base_for_build['base_value'] + base_for_build['ancestry_boost'] + base_for_build['class_boost']
        
        base_for_build['ability_modifier']= (base_for_build['total_boost'] -10)/2
        base_for_build= base_for_build.set_index('attribute')
        
        #create a refrence table that has this information for ALL class-ancestry combinations
        ability_mod_list= base_for_build['ability_modifier'].to_list()
        total_ability_list= base_for_build['total_boost'].to_list()

        #Add the final ability scores to a list to add to the main dataframe 
        build_list= build_list + ability_mod_list
        
        #add the calculated stats to the final dataframe of all build =s
        all_build_stats.loc[build_name]= build_list


#save the final dataframe as a out_data csv file 
all_build_stats.to_csv('../output_data/all_build_base_stats.csv')

#view final dataframe 
all_build_stats

,ancestry,class,level,strength,dexterity,constitution,intelligence,wisdom,charisma
dwarf barbarian strength,dwarf,barbarian strength,1,2.0,1.0,2.0,1.0,2.0,0.0
dwarf fighter strength,dwarf,fighter strength,1,2.0,1.0,2.0,1.0,2.0,0.0
dwarf fighter dexterity,dwarf,fighter dexterity,1,1.0,2.0,2.0,1.0,2.0,0.0
dwarf rogue dexterity,dwarf,rogue dexterity,1,1.0,2.0,2.0,1.0,2.0,0.0
dwarf sorcerer charisma,dwarf,sorcerer charisma,1,1.0,1.0,2.0,1.0,2.0,1.0
elf barbarian strength,elf,barbarian strength,1,2.0,2.0,0.0,2.0,1.0,1.0
elf fighter strength,elf,fighter strength,1,2.0,2.0,0.0,2.0,1.0,1.0
elf fighter dexterity,elf,fighter dexterity,1,1.0,3.0,0.0,2.0,1.0,1.0
elf rogue dexterity,elf,rogue dexterity,1,1.0,3.0,0.0,2.0,1.0,1.0
elf sorcerer charisma,elf,sorcerer charisma,1,1.0,2.0,0.0,2.0,1.0,2.0


--------

## open armor equipment list to be added as AC bonus to build

In [7]:
armor_list= pd.read_csv('../input_data/armor_equipment_list.csv')
armor_list.columns= armor_list.iloc[0]

#update number formatting for easier colculations
armor_list['Category'] = armor_list['Category'].str.lower()
armor_list['Strength']= pd.to_numeric(armor_list['Strength'], errors='coerce').fillna(0)
armor_list['AC Bonus']= pd.to_numeric(armor_list['AC Bonus'], errors='coerce').fillna(0)

#view dataframe
armor_list

,Name,Category,Level,Price,AC Bonus,Dex Cap,Check Penalty,Speed Penalty,Strength,Bulk,Group,Armor Traits
0,Name,category,Level,Price,0.0,Dex Cap,Check Penalty,Speed Penalty,0.0,Bulk,Group,Armor Traits
1,Unarmored,unarmored,—,—,0.0,NaN,—,—,0.0,—,—,—
2,Explorer's Clothing,unarmored,—,1 sp,0.0,+5,—,—,0.0,L,*Cloth*,*Comfort*
3,Padded Armor,light,—,2 sp,1.0,+3,—,—,10.0,L,*Cloth*,*Comfort*
4,Leather Armor,light,—,2 gp,1.0,+4,-1,—,10.0,1,*Leather*,—
5,Studded Leather Armor,light,—,3 gp,2.0,+3,-1,—,12.0,1,*Leather*,—
6,Chain Shirt,light,—,5 gp,2.0,+3,-1,—,12.0,1,*Chain*,"*Flexible*, *Noisy*"
7,Hide Armor,medium,—,2 gp,3.0,+2,-2,-5 ft.,14.0,2,*Leather*,—
8,Scale Mail,medium,—,4 gp,3.0,+2,-2,-5 ft.,14.0,2,*Composite*,—
9,Chain Mail,medium,—,6 gp,4.0,+1,-2,-5 ft.,16.0,2,*Chain*,"*Flexible*, *Noisy*"


In [8]:
weapon_list= pd.read_csv('../output_data/weapon_list_stats.csv')
weapon_list['Category'] = weapon_list['Category'].str.lower()
weapon_list

,Unnamed: 0,Name,Category,Level,Price,Damage,Bulk,Hands,Group,Weapon Traits,Damage_type
0,0,Fist,unarmed,0,0,1d4,0,1,Brawling,"Agile, Finesse, Nonlethal, Unarmed",B
1,1,Clan Dagger,simple,0,2 gp,1d4,L,1,Knife,"Agile, Dwarf, Parry, Uncommon, Versatile B",P
2,2,Club,simple,0,0,1d6,1,1,Club,Thrown 10 ft.,B
3,3,Dagger,simple,0,2 sp,1d4,L,1,Knife,"Agile, Finesse, Thrown 10 ft., Versatile S",P
4,4,Gauntlet,simple,0,2 sp,1d4,L,1,Brawling,"Agile, Free-Hand",B
...,...,...,...,...,...,...,...,...,...,...,...
99,99,Fire Poi,advanced,0,5 gp,1d4,L,1,Flail,"Agile, Backswing, Finesse, Twin, Uncommon",F
100,100,Gnome Flickmace,advanced,0,3 gp,1d8,2,1,Flail,"Gnome, Reach, Uncommon",B
101,101,Orc Necksplitter,advanced,0,2 gp,1d8,1,1,Axe,"Forceful, Orc, Sweep, Uncommon",S
102,102,Rhoka Sword,advanced,0,4 gp,1d8,2,1,Sword,"Deadly d8, Two-Hand 1d10, Uncommon",S


### Iterate between ancestry-class options and pull out all other skill and equipment boost scores

In [9]:
### create final dataframe for stats
column_list= ['ancestry', 'class', 'level'] + character_sheet.index.to_list()
all_build_skills=pd.DataFrame(columns=[column_list])

#create final dataframe for weapon damage output
weapon_column_list= ['ancestry', 'class', 'level', 'dmg_output_options']
weapon_options_build=pd.DataFrame(columns=[weapon_column_list])

#Create loop between class and ancestry options 
#first pull out the ancestry value 
for ancestry in all_ancestry:
    ancestry_choice= ancestry.lower().strip()
    
    #Then cycle through each class option for that ancestry 
    for build in range(len(chose_class_list)):
        class_choice= chose_class_list[build].lower().strip()
        print(class_choice)
        build_name= (ancestry_choice +' ' + class_choice)
        
        #copy base ability dataframe 
        skill_for_build= pd.DataFrame()

        #create list of first 3 column values
        skill_list=[]
        skill_list= [ancestry_choice, class_choice, 1]
        
        #input dictionary to convert the trained levels to prof numbers 
        prof_numbers= {'untrained': 0, 'trained': 2, 'expert': 4, 'master': 6, 'legendary': 8, 'Untrained': 0, 'Trained': 2, 'Expert': 4, 'Master': 6, 'Legendary': 8}
        
        ## first add all the ability scores to the 'ability_modifier' column 
        filter_ability= all_build_stats[(all_build_stats['ancestry'] == ancestry_choice) & (all_build_stats['class'] == class_choice)]
        #change the column names to the ability abreviations
        ability_abrev= {'strength': 'str', 'dexterity': 'dex', 'constitution':'con', 'intelligence':'int', 'wisdom': 'wis', 'charisma':'char'}
        filter_ability= filter_ability.rename(columns=ability_abrev)

        #make dictionary of values 
        skill_dict = filter_ability.T.to_dict()[build_name]
        
        ### add the values to the ability_mod column 
        skill_for_build['ability_modifier'] = character_sheet['ability_modifier'].replace(skill_dict)
        
        ####
        # Ancestry stat boosts
        ####
        #convert chosen ANCESTRY stats to a dictionary 
        ancestry_value= ancestry_stat[f'{ancestry_choice}'].to_dict()
        
        #replace the ancestry_boost values with the dictionary 
        skill_for_build['ancestry_boost'] =character_sheet['ancestry_boost'].replace(ancestry_value)

        ####
        #Class trained armor and weapon proficencies 
        ####
        #convert chosen class stats to a dictionary
        class_merge= class_base_stat[class_base_stat['class_build'] == chose_class_list[build]].T
        class_merge['metrics']= class_merge.index.str.lower()
        class_merge= class_merge.set_index('metrics', drop=True)
        class_merge= class_merge.rename(index={'hit points per level': 'hit points'})
        class_value= class_merge.to_dict()

        #replace the class_boost values with the class dictionary 
        skill_for_build['class_boost']= character_sheet['class_boost'].replace(class_value[class_merge.columns[0]])

        #pull out the trained levels for the armor and defense categories from the class dataframe 
        #trained_defense= class_merge.loc['defenses'].str.split(",")[-1]
        trained_defense= class_merge.loc['defenses'][0]
        trained_weapon= class_merge.loc['attacks'].str.split(",")[-1]
        #trained_weapon=class_merge.loc['defenses'][0]
        
        ## loop through the string of trained armor and defenses on the class info spreadsheet
        trained_weapon_defense=[]
        for x in trained_weapon:
            new_string= x.lower().strip() #.replace("'", '')
            trained_weapon_defense.append(new_string)
        for x in trained_defense:
            new_string= x.lower().strip()
            trained_weapon_defense.append(new_string)
        #print(trained_weapon_defense)
        
        #change the value in the class_boost to trained
        trained_weapon_sheet=[]
        for metric in character_sheet['class_boost']:
            if metric in trained_weapon_defense:
                trained_weapon_sheet.append('trained')
            else:
                trained_weapon_sheet.append(np.nan)

        #### Select out best armor option for the build, and add the armor bonus to the AC
        #filter armor by trained levels
        armor_options= armor_list[armor_list['Category'].isin(trained_weapon_defense)]

        #filter out armor that does not meet strength requirements
        strength_value= all_build_stats[all_build_stats.index == build_name]['strength'][0]+10
        #print("STRENGTH VALUE", strength_value)
        armor_options['Strength']= pd.to_numeric(armor_options['Strength'], errors='coerce').fillna(0)
        armor_options = armor_options[armor_options['Strength'] <= strength_value]
        
        #chose the armor with the HIGHEST AC bonus (ignorming other limitations like dex cap)
        armor_options['AC Bonus']= pd.to_numeric(armor_options['AC Bonus'], errors='coerce').fillna(0)
        armor_choice= armor_options.loc[armor_options['AC Bonus'].idxmax()]['Name']
        armor_ac_bonus=  armor_options.loc[armor_options['AC Bonus'].idxmax()]['AC Bonus']
        #print("BUILD ARMOR", build_name, armor_choice, armor_ac_bonus)

        #pull out weapon options based on the trained attacks, and create a list with the damage die options.
        #Option to refine these filters based on price and agile (etc) later
        weapon_options= weapon_list[weapon_list['Category'].isin(trained_weapon_defense)]
        dmg_die= weapon_options['Damage'].unique().tolist()
        
        base_list=[]
        base_list= [ancestry_choice, class_choice, 1]
        dmg_list= base_list + [dmg_die]
        #dmg_list= base_list + dmg_die

        #add the weapon options to the output build for action list 
        weapon_options_build.loc[build_name]= dmg_list
        
        
        #update the character sheet with the armor and weapon defenses 
        skill_for_build['proficiency_class']= trained_weapon_sheet
        skill_for_build['class_boost']= skill_for_build['class_boost'].replace(trained_weapon_sheet)
        skill_for_build['proficiency']= skill_for_build['proficiency_class'].fillna(skill_for_build['class_boost'])
        skill_for_build['proficiency']= skill_for_build['proficiency'].replace(prof_numbers)
        skill_for_build= skill_for_build.drop('proficiency_class', axis=1)

        #Add the armor AC bonus to the base ability_modifier column 
        skill_for_build['ability_modifier']= pd.to_numeric(skill_for_build['ability_modifier'], errors='coerce').fillna(0) #Make sure it is a number value first
        skill_for_build['ability_modifier']['armor class'] = skill_for_build['ability_modifier']['armor class'] + armor_ac_bonus
        #print("ADJUSTED AC", skill_for_build['ability_modifier']['armor class'])

        ####
        #fill all text values 
        ####
        skill_for_build['ability_modifier']= pd.to_numeric(skill_for_build['ability_modifier'], errors='coerce').fillna(0)
        skill_for_build['ancestry_boost']= pd.to_numeric(skill_for_build['ancestry_boost'], errors='coerce').fillna(0)
        skill_for_build['proficiency']= pd.to_numeric(skill_for_build['proficiency'], errors='coerce').fillna(0)
        
        # ### create a final column to add up all the bonuses 
        skill_for_build['level 1'] = character_sheet['base_value'] + skill_for_build['ability_modifier'] +skill_for_build['ancestry_boost'] +skill_for_build['proficiency']
        skill_list_level1=[]
        skill_list_level1= skill_for_build['level 1'].to_list()
        
        #Create a list of the metrics to save for each ancestry-class combination 
        skill_list_formerge=[]
        skill_list_formerge = skill_list + skill_list_level1

        #add the full list as a new row on the main dataframe 
        all_build_skills.loc[build_name]= skill_list_formerge 

#save dataframe in output_data folder 
all_build_skills.to_csv('../output_data/all_build_base_skills.csv')

#view final dataframe 
all_build_skills

barbarian strength
fighter strength
fighter   dexterity
rogue dexterity
sorcerer charisma
barbarian strength
fighter strength
fighter   dexterity
rogue dexterity
sorcerer charisma
barbarian strength
fighter strength
fighter   dexterity
rogue dexterity
sorcerer charisma
barbarian strength
fighter strength
fighter   dexterity
rogue dexterity
sorcerer charisma
barbarian strength
fighter strength
fighter   dexterity
rogue dexterity
sorcerer charisma
barbarian strength
fighter strength
fighter   dexterity
rogue dexterity
sorcerer charisma
barbarian strength
fighter strength
fighter   dexterity
rogue dexterity
sorcerer charisma


,ancestry,class,level,hit points,speed,perception,fortitude,reflex,will,armor class,...,medium,heavy,unarmed,simple,martial,advanced,other,magical tradition,spell attack,spell dc
dwarf barbarian strength,dwarf,barbarian strength,1,24.0,20.0,6.0,6.0,3.0,6.0,13.0,...,2.0,0.0,2.0,2.0,2.0,0.0,0.0,0.0,0.0,10.0
dwarf fighter strength,dwarf,fighter strength,1,22.0,20.0,6.0,6.0,5.0,4.0,13.0,...,2.0,2.0,2.0,0.0,2.0,0.0,0.0,0.0,0.0,10.0
dwarf fighter dexterity,dwarf,fighter dexterity,1,22.0,20.0,6.0,6.0,6.0,4.0,13.0,...,2.0,2.0,2.0,0.0,2.0,0.0,0.0,0.0,0.0,10.0
dwarf rogue dexterity,dwarf,rogue dexterity,1,20.0,20.0,6.0,4.0,6.0,6.0,13.0,...,0.0,0.0,2.0,2.0,0.0,0.0,0.0,0.0,0.0,10.0
dwarf sorcerer charisma,dwarf,sorcerer charisma,1,18.0,20.0,4.0,4.0,3.0,6.0,11.0,...,0.0,0.0,2.0,2.0,0.0,0.0,0.0,0.0,0.0,10.0
elf barbarian strength,elf,barbarian strength,1,18.0,30.0,5.0,4.0,4.0,5.0,14.0,...,2.0,0.0,2.0,2.0,2.0,0.0,0.0,0.0,0.0,10.0
elf fighter strength,elf,fighter strength,1,16.0,30.0,5.0,4.0,6.0,3.0,14.0,...,2.0,2.0,2.0,0.0,2.0,0.0,0.0,0.0,0.0,10.0
elf fighter dexterity,elf,fighter dexterity,1,16.0,30.0,5.0,4.0,7.0,3.0,14.0,...,2.0,2.0,2.0,0.0,2.0,0.0,0.0,0.0,0.0,10.0
elf rogue dexterity,elf,rogue dexterity,1,14.0,30.0,5.0,2.0,7.0,5.0,14.0,...,0.0,0.0,2.0,2.0,0.0,0.0,0.0,0.0,0.0,10.0
elf sorcerer charisma,elf,sorcerer charisma,1,12.0,30.0,3.0,2.0,4.0,5.0,12.0,...,0.0,0.0,2.0,2.0,0.0,0.0,0.0,0.0,0.0,10.0


In [10]:
#save weapon list damage output in dataframe 
weapon_options_build.to_csv('../output_data/weapon_damage_option_bybuild.csv')
weapon_options_build

,ancestry,class,level,dmg_output_options
dwarf barbarian strength,dwarf,barbarian strength,1,"[1d4, 1d6, 1d8, 1d10, 1d12]"
dwarf fighter strength,dwarf,fighter strength,1,"[1d4, 1d8, 1d10, 1d6, 1d12]"
dwarf fighter dexterity,dwarf,fighter dexterity,1,"[1d4, 1d8, 1d10, 1d6, 1d12]"
dwarf rogue dexterity,dwarf,rogue dexterity,1,"[1d4, 1d6, 1d8]"
dwarf sorcerer charisma,dwarf,sorcerer charisma,1,"[1d4, 1d6, 1d8]"
elf barbarian strength,elf,barbarian strength,1,"[1d4, 1d6, 1d8, 1d10, 1d12]"
elf fighter strength,elf,fighter strength,1,"[1d4, 1d8, 1d10, 1d6, 1d12]"
elf fighter dexterity,elf,fighter dexterity,1,"[1d4, 1d8, 1d10, 1d6, 1d12]"
elf rogue dexterity,elf,rogue dexterity,1,"[1d4, 1d6, 1d8]"
elf sorcerer charisma,elf,sorcerer charisma,1,"[1d4, 1d6, 1d8]"


-------------

## Next Steps
* Import class level-up features to update next level stat boosts
* Create an action-list for each build, using avaiable weapons, spells, and feats (optional later)
* Create a simulation for the average damage output per round 

-------------

### Attack action: 
* Chose weapon
* roll an attack roll (attack modifier for weapon) vs target AC

* Melee attack roll result = d20 roll + Strength modifier (or optionally Dexterity modifier for a finesse weapon) + proficiency bonus + other bonuses + penalties
* critical attack = double damage
* NOTE: Melee will add strength modifier UNLESS it is a finesse weapon (DEX modifier)

* Ranged attack roll result = d20 roll + Dexterity modifier + proficiency bonus + other bonuses + penalties

* Pentalites: The first is the multiple attack penalty, and the second is the range penalty.
* MAP: second attack (-5) OR (-4 if weapon is agile)

* Roll damage roll
* Melee damage roll = damage die of weapon or unarmed attack + Strength modifier + bonuses + penalties
* Ranged damage roll = damage die of weapon (+ Strength modifier for a thrown weapon or half Strength modifier for a propulsive weapon) + bonuses + penalties
* Spell (or similar effect) damage roll = damage die of effect + bonuses + penalties

Melee attack roll result = d20 roll + Strength modifier (or optionally Dexterity modifier for a finesse weapon) + proficiency bonus + other bonuses + penalties

Melee damage roll = damage die of weapon or unarmed attack + Strength modifier + bonuses + penalties

In [19]:
#### create action output dataframe 
action_columns=['build', 'level', 'action', 'attack_roll_result', 'attack modifier','penalties', 'dmg_dice', 'damage_type', 'damage_output', 'target_ac', 'crit_ac']
attack_action_bybuild= pd.DataFrame(columns = action_columns)

### loop through specific build options
for option in all_build_skills.index:
    build_choice= all_build_skills.loc[option]
    build_name= option
    build_choice_stats= all_build_stats.loc[option]
    build_weapon_choice= weapon_options_build.loc[option]
    
    #base target id
    target_ac= build_choice['armor class']
    
    #pull out the proficency bonus of the trained weapon 
    fitler_prof= pd.DataFrame(build_choice).T
    max_prof= fitler_prof[['unarmed', 'simple','martial','advanced','other']].max().max()
    
    #pull out attack roll modifiers
    str_attack = build_choice_stats['strength'] + max_prof
    str_dmg= build_choice_stats['strength']
    dex_attack= build_choice_stats['dexterity'] + max_prof
    dex_dmg= build_choice_stats['dexterity']
    
    #place holder column for the roll results
    d20 = 'random_roll'
    
    #for the given build, lop through all the weapon options and apply either the strength or dex attack bonus for action options
    #First create a list matching the column values of the output dataframe, THEN add it to the dataframe as a new row 
    for weapon in build_weapon_choice['dmg_output_options']:
        weapon_dmg_max= weapon.split('d')[-1]
        #If the character build is a STRENGTH build -> only str options
        if str_dmg > dex_dmg:
            output_row_str= [build_name, 1, f'str {weapon} melee attack 1', d20, str_attack, 0, weapon_dmg_max, 'melee',str_dmg, target_ac, target_ac+10] 
            attack_action_bybuild.loc[len(attack_action_bybuild)] = output_row_str
            output_row_str_2= [build_name, 1, f'str {weapon} melee attack 2', d20, str_attack, 5, weapon_dmg_max, 'melee',str_dmg, target_ac, target_ac+10]
            attack_action_bybuild.loc[len(attack_action_bybuild)] = output_row_str_2
          
        #If the character build is a DEX build -> only DEX options
        if str_dmg < dex_dmg:
            output_row_dex= [build_name, 1, f'dex {weapon} melee attack 1', d20, dex_attack, 0, weapon_dmg_max, 'melee',dex_dmg, target_ac, target_ac+10]
            attack_action_bybuild.loc[len(attack_action_bybuild)] =output_row_dex
            output_row_dex_2= [build_name, 1, f'dex {weapon} melee attack 2', d20, dex_attack, 5, weapon_dmg_max, 'melee',dex_dmg, target_ac, target_ac+10]
            attack_action_bybuild.loc[len(attack_action_bybuild)] = output_row_dex_2

        #If the character build is equal in both, have all options
        if str_dmg == dex_dmg:
            output_row_str= [build_name, 1, f'str {weapon} melee attack 1', d20, str_attack, 0, weapon_dmg_max, 'melee',str_dmg, target_ac, target_ac+10] 
            attack_action_bybuild.loc[len(attack_action_bybuild)] = output_row_str
            output_row_str_2= [build_name, 1, f'str {weapon} melee attack 2', d20, str_attack, 5, weapon_dmg_max, 'melee',str_dmg, target_ac, target_ac+10]
            attack_action_bybuild.loc[len(attack_action_bybuild)] = output_row_str_2
            output_row_dex= [build_name, 1, f'dex {weapon} melee attack 1', d20, dex_attack, 0, weapon_dmg_max, 'melee',dex_dmg, target_ac, target_ac+10]
            attack_action_bybuild.loc[len(attack_action_bybuild)] =output_row_dex
            output_row_dex_2= [build_name, 1, f'dex {weapon} melee attack 2', d20, dex_attack, 5, weapon_dmg_max, 'melee',dex_dmg, target_ac, target_ac+10]
            attack_action_bybuild.loc[len(attack_action_bybuild)] = output_row_dex_2
                
attack_action_bybuild  

### save melee attack option as a dataframe 
attack_action_bybuild.to_csv('../output_data/melee_attack_output_all-builds.csv')
attack_action_bybuild  

,build,level,action,attack_roll_result,attack modifier,penalties,dmg_dice,damage_type,damage_output,target_ac,crit_ac
0,dwarf barbarian strength,1,str 1d4 melee attack 1,random_roll,4.0,0,4,melee,2.0,13.0,23.0
1,dwarf barbarian strength,1,str 1d4 melee attack 2,random_roll,4.0,5,4,melee,2.0,13.0,23.0
2,dwarf barbarian strength,1,str 1d6 melee attack 1,random_roll,4.0,0,6,melee,2.0,13.0,23.0
3,dwarf barbarian strength,1,str 1d6 melee attack 2,random_roll,4.0,5,6,melee,2.0,13.0,23.0
4,dwarf barbarian strength,1,str 1d8 melee attack 1,random_roll,4.0,0,8,melee,2.0,13.0,23.0
...,...,...,...,...,...,...,...,...,...,...,...
387,orc-dex sorcerer charisma,1,dex 1d4 melee attack 2,random_roll,4.0,5,4,melee,2.0,12.0,22.0
388,orc-dex sorcerer charisma,1,dex 1d6 melee attack 1,random_roll,4.0,0,6,melee,2.0,12.0,22.0
389,orc-dex sorcerer charisma,1,dex 1d6 melee attack 2,random_roll,4.0,5,6,melee,2.0,12.0,22.0
390,orc-dex sorcerer charisma,1,dex 1d8 melee attack 1,random_roll,4.0,0,8,melee,2.0,12.0,22.0
